8

In [1]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=10000, noise=0.4, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

In [2]:
from sklearn.model_selection import ShuffleSplit

n_trees = 1000
n_instances = 100

mini_sets = []

rs = ShuffleSplit(n_splits=n_trees, test_size=len(X_train) - n_instances, random_state=42)

for mini_train_index, mini_test_index in rs.split(X_train):
    X_mini_train = X_train[mini_train_index]
    y_mini_train = y_train[mini_train_index]
    mini_sets.append((X_mini_train, y_mini_train))

In [3]:
mini_sets[0]

(array([[-3.15325488e-01,  4.94322659e-01],
        [ 1.07395888e+00, -3.83006874e-01],
        [ 1.23368080e+00, -2.02727544e-01],
        [ 1.45327595e+00, -4.97650493e-01],
        [ 6.29403125e-01, -4.58057181e-01],
        [ 1.31621613e+00, -4.96340629e-01],
        [ 6.61605020e-01, -5.25120656e-01],
        [ 1.17772151e+00,  2.12896730e-01],
        [ 1.27074026e+00,  8.37618484e-01],
        [ 2.40777741e-01, -4.05280317e-01],
        [ 2.09601211e+00,  2.29976411e-01],
        [ 3.96488095e-01, -3.07170096e-01],
        [-7.76673348e-01,  1.98996929e-01],
        [-8.88960092e-01,  1.06022488e+00],
        [ 1.21458297e+00, -3.55536724e-01],
        [ 1.72720880e+00,  3.94406303e-01],
        [ 1.96647532e+00,  2.91551342e-01],
        [-2.00602385e-01,  4.62530417e-02],
        [-1.19424550e+00,  1.64842526e-01],
        [-2.98171028e-01,  4.07082497e-01],
        [ 2.07327094e+00,  2.02547469e-02],
        [-7.43875589e-01,  1.01769167e+00],
        [-1.03747473e+00,  4.298

In [4]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone
from sklearn.metrics import accuracy_score
import numpy as np

tree = DecisionTreeClassifier(
    max_depth=6, 
    max_leaf_nodes=15, 
    random_state=42
)

In [5]:
forest = [clone(tree) for _ in range(n_trees)]

accuracy_scores = []

for tree, (X_mini_train, y_mini_train) in zip(forest, mini_sets):
    tree.fit(X_mini_train, y_mini_train)

    y_pred = tree.predict(X_test)
    accuracy_scores.append(accuracy_score(y_test, y_pred)) 

np.mean(accuracy_scores)

np.float64(0.8076169999999999)

In [6]:
Y_pred = np.empty([n_trees, len(X_test)], dtype=np.uint8)

for tree_index, tree in enumerate(forest):
    Y_pred[tree_index] = tree.predict(X_test)

In [7]:
Y_pred

array([[0, 1, 0, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 0, ..., 1, 0, 0],
       ...,
       [1, 1, 0, ..., 0, 0, 0],
       [1, 1, 0, ..., 0, 0, 1],
       [1, 1, 0, ..., 0, 0, 0]], shape=(1000, 2000), dtype=uint8)

In [8]:
from scipy.stats import mode

y_pred_majority_votes, n_votes = mode(Y_pred, axis=0)

In [9]:
accuracy_score(y_test, y_pred_majority_votes.reshape([-1]))

0.8735